# Phase 3 — Hybrid Retrieval Validation

Compares **BM25** (keyword) vs **vector** (semantic) vs **hybrid** (RRF fusion) over the code index.

The embedder is pluggable via `EMBEDDING_PROVIDER` in `.env` — default **local** (fastembed BGE-small, offline), or **voyage** (voyage-code-3).

**Run first:** `uv run python -m archaeologist.indexing.run`

Hybrid should be robust where each half is weak: BM25 misses synonyms, vectors miss exact identifiers — RRF blends both.

In [ ]:
import os, sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))

from archaeologist.config import settings
from archaeologist.indexing import code_index
from archaeologist.indexing.opensearch_client import get_client
from archaeologist.retrieval.embeddings import get_embedder
from archaeologist.retrieval.hybrid import hybrid_search

client = get_client()
embedder = get_embedder()
n_vec = client.count(index=code_index.SYMBOL_INDEX, body={"query": {"exists": {"field": "embedding"}}})["count"]
print("provider:", settings.embedding_provider)
print("embedder:", type(embedder).__name__ if embedder else "NONE")
print("symbols with vectors:", n_vec)

In [ ]:
def line(src, score):
    loc = f"{src['file_path']}:{src['start_line']}"
    qn = src['qualified_name'] or src['name']
    return f"    {score:.4f}  [{src['kind']:8}] {qn:36.36} {loc}"

def compare(query, k=5):
    print(f"\n########## {query!r}\n")
    print("  BM25:")
    for _id, src in code_index.bm25_hits(client, query, k):
        print(line(src, 0.0))
    if embedder:
        print("  VECTOR:")
        vec = embedder.embed_query(query)
        for _id, src in code_index.knn_hits(client, vec, k):
            print(line(src, 0.0))
    print("  HYBRID (RRF):")
    for hit in hybrid_search(client, embedder, query, k=k):
        print(line(hit, hit['score']))

## Side-by-side on questions phrased in natural language

In [ ]:
compare("how does Flask decide which view handles an incoming request")
compare("turning a Python object into an HTTP response body")
compare("registering code to run before each request")

## Summary

In [ ]:
total = client.count(index=code_index.SYMBOL_INDEX)["count"]
ok = bool(embedder) and n_vec > 0
print("Phase 3 —", "HYBRID OK ✅" if ok else "BM25-ONLY (no embedder / reindex needed) ⚠️")
print(f"  provider          : {settings.embedding_provider}")
print(f"  indexed symbols   : {total}")
print(f"  with vectors      : {n_vec}")